# Modules & Crates

Welcome! As your Rust projects grow, you need a way to organize code. **Modules** let you group related items and control visibility. **Crates** are the compilation units — libraries or executables. This guide covers the module system, visibility, paths, external crates, and Cargo basics.

## What Are Modules and Crates?

Think of a **crate** as a package (like a box). A **module** is a compartment inside that box. The crate root (e.g. `main.rs` or `lib.rs`) is the entry point. Modules let you split code into logical units and hide implementation details.

## The mod Keyword

Use `mod` to declare a module. Modules can be declared inline (with curly braces) or in a separate file.

In [ ]:
mod math {
    pub fn add(a: i32, b: i32) -> i32 {
        a + b
    }
    fn private_helper() -> i32 {
        42
    }
}

// add is public, we can use it
println!("math::add(1, 2) = {}", math::add(1, 2));
// math::private_helper();  // error: private

## Nested Modules

Modules can contain other modules. Use `::` to navigate the hierarchy.

In [ ]:
mod outer {
    pub mod inner {
        pub fn greet() {
            println!("Hello from inner!");
        }
    }
}

outer::inner::greet();

## pub Visibility

By default, items are **private** to their module. Use `pub` to make them visible to parent modules and beyond. `pub(crate)` limits visibility to the current crate.

In [ ]:
mod visibility {
    pub fn public_fn() {}
    fn private_fn() {}
    pub struct PublicStruct {
        pub field: i32,   // field also needs pub to be accessible
    }
}

let s = visibility::PublicStruct { field: 10 };
println!("field: {}", s.field);

## use Keyword and Paths

The `use` keyword brings items into scope so you don't have to type the full path every time.

In [ ]:
use std::collections::HashMap;
let mut map = HashMap::new();
map.insert("key", 1);
println!("map: {:?}", map);

## Path Prefixes: crate::, super::, self::

- `crate::` — from the crate root
- `super::` — from the parent module
- `self::` — from the current module (usually omitted)

In [ ]:
mod a {
    pub fn from_a() { println!("from_a"); }
    pub mod b {
        pub fn from_b() {
            super::from_a();  // access parent
        }
    }
}

a::b::from_b();

## Re-exporting with pub use

`pub use` re-exports an item, making it part of your module's public API. Useful for creating a clean facade.

In [ ]:
mod backend {
    pub fn internal() { println!("internal"); }
}

mod api {
    pub use crate::backend::internal as do_work;  // re-export with new name
}

api::do_work();

## Separating Modules into Files

In a real project: `mod foo;` tells the compiler to look for `foo.rs` or `foo/mod.rs`. The file's contents become the module body. No need to wrap in `mod foo { ... }` — the file itself is the module.

Example structure:
```
src/
  main.rs      // mod utils;
  utils.rs     // pub fn helper() { ... }
  utils/
    mod.rs     // pub mod sub;
    sub.rs     // pub fn sub_helper() { ... }
```

## External Crates (Cargo.toml Dependencies)

Add dependencies in `Cargo.toml` under `[dependencies]`. Then `use` them in your code. Cargo downloads and compiles them.

```toml
[dependencies]
serde = { version = "1.0", features = ["derive"] }
tokio = { version = "1", features = ["full"] }
rand = "0.8"
```

## The Prelude

Rust automatically brings common items into scope via the **prelude** (`std::prelude::v1`). That's why you can use `Vec`, `Option`, `Result`, `println!`, etc. without `use`.

## Commonly Used Crates

| Crate | Purpose |
|-------|--------|
| **serde** | Serialization/deserialization (JSON, etc.) |
| **tokio** | Async runtime |
| **clap** | Command-line argument parsing |
| **rand** | Random number generation |
| **reqwest** | HTTP client |
| **thiserror / anyhow** | Error handling |

In [ ]:
// Example: rand (if added to Cargo.toml)
// use rand::Rng;
// let n: u32 = rand::thread_rng().gen_range(1..=100);
// In a notebook, external crates may need to be available in the environment

## Cargo Basics

| Command | Purpose |
|---------|--------|
| `cargo new project_name` | Create a new binary project |
| `cargo new --lib lib_name` | Create a library |
| `cargo build` | Compile (debug) |
| `cargo build --release` | Optimized build |
| `cargo run` | Build and run |
| `cargo test` | Run tests |
| `cargo doc --open` | Generate and open docs |

## Workspaces

A **workspace** is a set of crates that share a `Cargo.lock` and output directory. Define in the root `Cargo.toml`:

```toml
[workspace]
members = ["crate_a", "crate_b"]
```

## Feature Flags

Features let you enable optional functionality. In `Cargo.toml`:

```toml
[features]
default = ["std"]
std = []
serde = ["dep:serde"]
```

Use with `cargo build --features serde` or in dependencies: `mylib = { path = ".", features = ["serde"] }`.

## Common Mistakes Beginners Make

1. **Forgetting `pub`** on items you want to use from outside the module
2. **Confusing `mod` and `use`** — `mod` declares/loads; `use` brings into scope
3. **Wrong path** — `crate::` vs `super::` vs relative paths
4. **Circular dependencies** — avoid modules depending on each other in a cycle
5. **Not running `cargo build`** after adding dependencies

## Key Rules to Remember

- `mod` declares a module; `pub` makes items visible
- `use` brings items into scope; `pub use` re-exports
- Paths: `crate::` (root), `super::` (parent), `self::` (current)
- `mod foo;` loads `foo.rs` or `foo/mod.rs`
- Dependencies go in `Cargo.toml`; `cargo build` fetches them
- Workspaces group multiple crates; features enable optional code